## Setup

In [ ]:
import numpy as np
import pandas as pd
import torch
import optuna
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import classification_report
from lightgbm import LGBMClassifier
from pytorch_tabnet.tab_model import TabNetClassifier

from src.data_loading import load_class_data
from src.eda_utils import remove_correlated_features, seleziona_variabili_pca
from src.preprocessing import build_preprocessing_pipeline, preprocess_arrays
from src.evaluation import evaluate_model

In [ ]:
TRAIN_FEATURES_PATH = "../data/train_features_bone_marrow.h5"
TEST_FEATURES_PATH = "../data/test_features_bone_marrow.h5"

X_train, y_train = load_class_data(TRAIN_FEATURES_PATH)
X_test, y_test = load_class_data(TEST_FEATURES_PATH)

df_train = pd.DataFrame(X_train)
df_test = pd.DataFrame(X_test)
df_train[64] = y_train
df_test[64] = y_test
df_train[64] = df_train[64].astype('category').cat.codes
df_test[64] = df_test[64].astype('category').cat.codes

In [ ]:
# Stessa riduzione a 30 variabili usata in 02_model_comparison.ipynb
X_train_filtered, to_drop = remove_correlated_features(df_train.iloc[:, :-1], threshold=0.5)
df_filtered = df_train.drop(columns=to_drop)

df_ridotto = seleziona_variabili_pca(df_filtered)
column_names = df_ridotto.columns.tolist()

df_train = df_train.loc[:, df_train.columns.isin(column_names)]
df_test = df_test.loc[:, df_test.columns.isin(column_names)]

X_train = df_train.iloc[:, :-1]
y_train = df_train.iloc[:, -1]
X_test = df_test.iloc[:, :-1]
y_test = df_test.iloc[:, -1]
numeric_columns = X_train.columns.tolist()

sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

##k-fold cross validation
splitta il training set in k partizioni e crea k versioni del modello composto ogni volta da k-1 partizioni e poi questi k modelli vengono testati sulla partizione che è rimasta fuori dal modello.

La media e la deviazione standard dei punteggi k risultanti forniranno una stima e una quantificazione del livello di incertezza, da utilizzare come stima del modello per le prestazioni attese su dati futuri non osservati.

La scelta di k è molto importante; generalmente si sceglie k = 5 o k = 10, con k = 10 per alta precisione e k = 5 che è un buon compromesso tra precisione e costo computazionale.

#TUNING DEI PARAMETRI

### Tuning LightGBM con GridSearchCV

In [ ]:
lgbm_model = LGBMClassifier(
    objective='multiclass',
    num_class=5,
    eval_metric='multi_logloss',
    random_state=0,
)

lgbm_pipeline = Pipeline([
    ('processing', build_preprocessing_pipeline(numeric_columns)),
    ('modeling', lgbm_model)
])

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Definizione della griglia degli iperparametri
param_grid = {
    'modeling__n_estimators': [100, 200, 300],
    'modeling__max_depth': [4, 6, 8],
    'modeling__learning_rate': [0.01, 0.05, 0.1],
    'modeling__min_child_samples': [10, 20, 30],  # in LightGBM si chiama così
}

# Imposta la cross-validation stratificata
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Oggetto GridSearchCV
grid_search = GridSearchCV(
    estimator=lgbm_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='f1_macro',  # puoi cambiare con 'accuracy', 'balanced_accuracy' ecc.
    n_jobs=-1,            # parallelizza su tutti i core
    verbose=2,
)

# Esegui Grid Search con sample_weight
grid_search.fit(X_train, y_train, modeling__sample_weight=sample_weights)

# Migliori parametri
print("\nMigliori parametri trovati:")
print(grid_search.best_params_)

# Miglior punteggio
print(f"\nMiglior punteggio f1_macro: {grid_search.best_score_:.4f}")


Fitting 5 folds for each of 81 candidates, totalling 405 fits


/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: use_label_encoder
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=10, will be overridden by min_samples_leaf=3. Current value: min_data_in_leaf=3
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: use_label_encoder
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=10, will be overridden by min_samples_leaf=3. Current value: min_data_in_leaf=3
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014884 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 7650
[LightGBM] [Info] Number of data points in the train set: 44800, number of used features: 30
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training 

In [ ]:
best_model = grid_search.best_estimator_


In [ ]:
y_pred = best_model.predict(X_test)
lgbm_tuned_metrics = evaluate_model(y_test, y_pred, model_name="LightGBM (tuned)")

/usr/local/lib/python3.11/dist-packages/sklearn/utils/deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: use_label_encoder
[LightGBM] [Warning] min_data_in_leaf is set with min_child_samples=10, will be overridden by min_samples_leaf=3. Current value: min_data_in_leaf=3
              precision    recall  f1-score   support

           0       0.48      0.10      0.16       132
           1       0.72      0.82      0.77      1765
           2       0.89      0.91      0.90      8219
           3       0.88      0.86      0.87      7873
           4       0.66      0.57      0.61      1212

    accuracy                           0.85     19201
   macro avg       0.73      0.65      0.66     19201
weighted avg       0.85      0.85      0.85     19201



### Tuning TabNet con Optuna

In [ ]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 12.2 MB/s eta 0:00:00


In [ ]:
# Preprocessing per TabNet (richiede array numpy, non una Pipeline sklearn)
X_train_arr, X_test_arr = preprocess_arrays(X_train.values, X_test.values)
y_train_arr = y_train.values
y_test_arr = y_test.values

In [ ]:
import optuna
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

def objective(trial):
    # Parametri da ottimizzare
    n_d = trial.suggest_int("n_d", 8, 64, step=8)
    n_a = trial.suggest_int("n_a", 8, 64, step=8)
    n_steps = trial.suggest_int("n_steps", 3, 10)
    gamma = trial.suggest_float("gamma", 1.0, 2.0)
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)
    batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
    virtual_batch_size = trial.suggest_categorical("virtual_batch_size", [64, 128, 256])

    model = TabNetClassifier(
        n_d=n_d,
        n_a=n_a,
        n_steps=n_steps,
        gamma=gamma,
        optimizer_params=dict(lr=lr),
        scheduler_params={"step_size":50, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='entmax',
        verbose=0,
        device_name='cuda' if torch.cuda.is_available() else 'cpu'
    )

    model.fit(
        X_train=X_train_arr,
        y_train=y_train_arr,
        eval_set=[(X_test_arr, y_test_arr)],
        eval_name=['val'],
        eval_metric=['balanced_accuracy'],
        max_epochs=100,
        patience=10,
        batch_size=batch_size,
        virtual_batch_size=virtual_batch_size,
        num_workers=0,
        drop_last=False,
        weights=sample_weights
    )

    preds = model.predict(X_test_arr)
    score = balanced_accuracy_score(y_test_arr, preds)
    return score

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

# Migliori iperparametri trovati
print("Best parameters:", study.best_params)


[I 2025-06-25 13:56:57,252] A new study created in memory with name: no-name-a5223d7f-deb3-4ab6-86ec-b4beae565c37
/tmp/ipython-input-19-3811182922.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)



Early stopping occurred at epoch 17 with best_epoch = 7 and best_val_balanced_accuracy = 0.68547


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2025-06-25 14:02:50,478] Trial 0 finished with value: 0.6854741076958712 and parameters: {'n_d': 32, 'n_a': 64, 'n_steps': 5, 'gamma': 1.1932922102299064, 'lr': 0.032406076042113485, 'batch_size': 256, 'virtual_batch_size': 64}. Best is trial 0 with value: 0.6854741076958712.
/tmp/ipython-input-19-3811182922.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)



Early stopping occurred at epoch 31 with best_epoch = 21 and best_val_balanced_accuracy = 0.70519


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2025-06-25 14:09:57,532] Trial 1 finished with value: 0.705185673147116 and parameters: {'n_d': 64, 'n_a': 16, 'n_steps': 4, 'gamma': 1.5577795292973309, 'lr': 0.03198670750566383, 'batch_size': 256, 'virtual_batch_size': 64}. Best is trial 1 with value: 0.705185673147116.
/tmp/ipython-input-19-3811182922.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)


...


Early stopping occurred at epoch 28 with best_epoch = 18 and best_val_balanced_accuracy = 0.67522


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2025-06-25 18:10:37,541] Trial 28 finished with value: 0.6752203321570869 and parameters: {'n_d': 32, 'n_a': 24, 'n_steps': 5, 'gamma': 1.962716367651531, 'lr': 0.09426761682984477, 'batch_size': 512, 'virtual_batch_size': 128}. Best is trial 26 with value: 0.7226926973469505.
/tmp/ipython-input-19-3811182922.py:13: FutureWarning: suggest_loguniform has been deprecated in v3.0.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v3.0.0. Use suggest_float(..., log=True) instead.
  lr = trial.suggest_loguniform("lr", 1e-4, 1e-1)



Early stopping occurred at epoch 42 with best_epoch = 32 and best_val_balanced_accuracy = 0.70729


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2025-06-25 18:21:07,759] Trial 29 finished with value: 0.7072910795546223 and parameters: {'n_d': 32, 'n_a': 32, 'n_steps': 7, 'gamma': 1.8047468864168656, 'lr': 0.039299067129195044, 'batch_size': 512, 'virtual_batch_size': 64}. Best is trial 26 with value: 0.7226926973469505.


Best parameters: {'n_d': 32, 'n_a': 40, 'n_steps': 6, 'gamma': 1.8458143667544906, 'lr': 0.050673769152653377, 'batch_size': 512, 'virtual_batch_size': 64}


### Modello finale TabNet con i migliori iperparametri

In [ ]:
tabnet_model = TabNetClassifier(
    n_d=32,
    n_a=40,
    n_steps=6,
    gamma=1.8458143667544906,
    n_independent=2,
    n_shared=2,
    optimizer_params=dict(lr=0.050673769152653377),
    scheduler_params={"step_size": 50, "gamma": 0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',
    seed=0,
    verbose=10,
    device_name='cuda' if torch.cuda.is_available() else 'cpu',
)

tabnet_model.fit(
    X_train=X_train_arr,
    y_train=y_train_arr,
    eval_set=[(X_test_arr, y_test_arr)],
    eval_name=['val'],
    eval_metric=['balanced_accuracy'],
    max_epochs=200,
    patience=20,
    batch_size=512,
    virtual_batch_size=64,
    num_workers=0,
    drop_last=False,
    weights=sample_weights
)

/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 1.41277 | val_balanced_accuracy: 0.48856 |  0:00:19s
epoch 10 | loss: 0.97277 | val_balanced_accuracy: 0.66115 |  0:02:40s
epoch 20 | loss: 0.67663 | val_balanced_accuracy: 0.68173 |  0:04:49s
epoch 30 | loss: 0.56447 | val_balanced_accuracy: 0.70457 |  0:06:57s
epoch 40 | loss: 0.49652 | val_balanced_accuracy: 0.6879  |  0:09:08s

Early stopping occurred at epoch 49 with best_epoch = 29 and best_val_balanced_accuracy = 0.72269


/usr/local/lib/python3.11/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [ ]:
preds = tabnet_model.predict(X_test_arr)
tabnet_tuned_metrics = evaluate_model(y_test_arr, preds, model_name="TabNet (tuned)")


Classification Report:
              precision    recall  f1-score   support

           0       0.13      0.53      0.21       132
           1       0.69      0.69      0.69      1765
           2       0.91      0.85      0.88      8219
           3       0.87      0.85      0.86      7873
           4       0.54      0.70      0.61      1212

    accuracy                           0.82     19201
   macro avg       0.63      0.72      0.65     19201
weighted avg       0.84      0.82      0.83     19201

Balanced Accuracy: 0.7226926973469505
F1 Macro: 0.6482818377938713
Accuracy: 0.8205822613405552
Recall Macro: 0.723
